# 02 — Pose Baselines and Latency Diagnostics

This notebook evaluates pose estimation and person detection candidates against the fixed `development` protocol. The `mediapipe_opencv` candidate downloads (if needed), hash-verifies, and loads the real pinned ONNX weights via `cv2.dnn.readNetFromONNX`, running genuine `net.forward()` inference — no golden/fixed tensors are used for evaluation.

No authorized recorded clips exist in this repository yet, so this notebook currently runs in **explicit synthetic mode** (`allow_synthetic=True`): clip frames are synthetic and every clip evaluated must declare `consent_scope: "synthetic-no-person"`. Synthetic mode validates mechanics and real model latency only — coverage and confidence numbers are not meaningful (the synthetic imagery contains no real person) and must never be reported as real evaluation results. Once authorized clips exist, set `BENCHMARK_ALLOW_SYNTHETIC=false` and `DATASET_MEDIA_ROOT` to the authorized media directory to run a real evaluation; the benchmark fails clearly if authorized media is unavailable rather than silently substituting synthetic frames.

It imports reusable evaluation functions from `kinetiq_v_vision.evaluation.baselines` and asserts benchmark invariants programmatically. In accordance with project policy, no metrics are fabricated and no bulky execution outputs or raw weights are committed to version control.

In [ ]:
import os
from pathlib import Path

from kinetiq_v_vision.evaluation.baselines import run_pose_baseline_benchmark

# Resolve manifest path from environment or fall back to verified synthetic CI fixture
DEFAULT_FIXTURE = Path("../fixtures/data/synthetic_manifest.json").resolve()
manifest_path = os.getenv("DATASET_MANIFEST_PATH", str(DEFAULT_FIXTURE))
split = os.getenv("BENCHMARK_SPLIT", "development")

# No authorized recorded clips exist yet, so synthetic mode defaults to on.
# Set BENCHMARK_ALLOW_SYNTHETIC=false (and DATASET_MEDIA_ROOT to an authorized
# media directory) once real clips are available to run a real evaluation.
allow_synthetic = os.getenv("BENCHMARK_ALLOW_SYNTHETIC", "true").lower() != "false"
media_root = os.getenv("DATASET_MEDIA_ROOT")

print(f"Benchmarking pose candidates on '{split}' split from: {manifest_path}")
print(f"Mode: {'SYNTHETIC (mechanics/latency only)' if allow_synthetic else 'REAL (authorized media required)'}")

In [ ]:
report = run_pose_baseline_benchmark(
    manifest_path=manifest_path,
    split=split,
    max_frames_per_clip=10,
    warmup_frames=2,
    media_root=media_root,
    allow_synthetic=allow_synthetic,
)
print(report.summary_markdown())

In [ ]:
# Programmatic verification of benchmark invariants
assert report.total_clips_evaluated > 0, "No clips evaluated in benchmark"
assert len(report.candidate_metrics) >= 1, "No candidates evaluated"
assert report.environment.is_synthetic == allow_synthetic, "Synthetic labeling must match the requested mode"

for cid, metrics in report.candidate_metrics.items():
    assert metrics.total_frames > 0, f"Candidate {cid} evaluated zero frames"
    assert metrics.p50_latency_ms > 0.0, f"Candidate {cid} p50 latency must be strictly positive"
    assert metrics.p95_latency_ms >= metrics.p50_latency_ms, f"Candidate {cid} p95 must be >= p50"
    assert metrics.throughput_fps > 0.0, f"Candidate {cid} throughput FPS must be positive"
    assert 0.0 <= metrics.mean_landmark_confidence <= 1.0, f"Candidate {cid} confidence out of bounds"

    if not report.environment.is_synthetic:
        # Only meaningful for a real evaluation on authorized recorded movement.
        assert metrics.person_coverage_pct > 0.0, f"Candidate {cid} has zero person detection coverage"
        assert metrics.pose_coverage_pct > 0.0, f"Candidate {cid} has zero pose landmark coverage"

if report.environment.is_synthetic:
    print(
        "Dataset governance: synthetic-mode mechanics and real model latency verified. "
        "Coverage numbers are not meaningful on synthetic imagery and are not asserted here — "
        "a real evaluation requires authorized recorded clips (BENCHMARK_ALLOW_SYNTHETIC=false)."
    )
else:
    print("Pose baselines benchmark: all invariants verified on real authorized media.")